# Changes in Global Energy Supply and Source Shares

**San Diego State University · STAT 420 Data Curation and Management · Fall 2026**

Kevin Schwartz, Mariana Khoshaba, and Jason Wu  
Instructor Ryan Paul Lafler

Run the pipeline once from the repository root before running this notebook. See the [README](../README.md#reproduce-the-analysis) for PostgreSQL and SQLite instructions. The notebook works from the root or the `notebooks` directory and reads the chosen engine's exports. Its executed copy can be exported with `python -m src.export_completed_notebook OUTPUT_DIRECTORY`.

The original core implementation was published at commit `4c65ccc4bb4c6990d6f66ce9d3abce5baf530d8e`. Mariana Khoshaba consolidated that implementation and developed the geographic and historical visualizations. The current edition validates the coverage and historical inputs before displaying those extensions.

# Part I — Original Team Implementation

## 1. Environment and Project Setup

The project uses a Python 3.11 environment with repository paths configured for the data, source code, SQL, results, figures, documentation, and tests.

In [ ]:
from pathlib import Path
import json
import platform
import sys

import pandas as pd
import plotly
from IPython.display import Image, display

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (current, *current.parents)
     if (folder / "src/config.py").is_file() and (folder / "sql/queries.sql").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project root or its notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CATEGORIES, FIGURES as FIGURES_DIR, PROCESSED as PROCESSED_DIR
from src.config import RAW as RAW_DIR, RESULTS as RESULTS_DIR, TOTAL
from src.extended_analysis import (
    MAP_START, MAP_END, RENEWABLE_TOLERANCE_TWH, geographic_panel,
    geographic_coverage, geographic_figures, historical_figures, historical_values, map_codes,
)
from src.make_extension_figures import make_extension_figures

SQL_DIR = PROJECT_ROOT / "sql"
SRC_DIR = PROJECT_ROOT / "src"
TESTS_DIR = PROJECT_ROOT / "tests"
DOCS_DIR = PROJECT_ROOT / "docs"
print(f"Python {platform.python_version()}, pandas {pd.__version__}, Plotly {plotly.__version__}")

The project requirements are pinned for Python 3.11.

In [ ]:
print((PROJECT_ROOT / "requirements.txt").read_text())

## 2. Project Configuration

**Original repository file:** `src/config.py`

Defines the fixed OWID data release, project paths, analysis period, selected geographies, and energy indicators used throughout the workflow.

In [ ]:
print((SRC_DIR / "config.py").read_text(encoding="utf-8"))

## 3. Data Download

**Original repository file:** `src/download_data.py`

Downloads the pinned Our World in Data (OWID) Energy dataset and verifies the source file before analysis.

In [ ]:
print((SRC_DIR / "download_data.py").read_text(encoding="utf-8"))

## 4. Raw Data Profiling

**Original repository file:** `src/profile_data.py`

Profiles the raw OWID dataset for coverage, missing values, duplicates, and data quality issues.

In [ ]:
print(
    (SRC_DIR / "profile_data.py")
    .read_text(encoding="utf-8")
)

## 5. Data Cleaning and Curation

**Original repository file:** `src/clean_data.py`

Validates the selected indicators and reshapes the raw data into the project's curated relational tables.

In [ ]:
print(
    (SRC_DIR / "clean_data.py")
    .read_text(encoding="utf-8")
)

## 6. Relational Database Schema

**Original repository file:** `sql/schema.sql`

Defines the relational database structure for source releases, geographies, indicators, and energy observations.

In [ ]:
print((SQL_DIR / "schema.sql").read_text(encoding="utf-8"))

## 7. Database Loading

**Original repository file:** `src/load_database.py`

Loads the curated tables into the selected database, validates relational integrity, and executes the project SQL queries.

In [ ]:
print((SRC_DIR / "load_database.py").read_text(encoding="utf-8"))

## 8. SQL Queries

**Original repository file:** `sql/queries.sql`

Retrieves the World Series, annual coverage, and selected-country series used in the analysis.

In [ ]:
print((SQL_DIR / "queries.sql").read_text(encoding="utf-8"))

## 9. Statistical Analysis

**Original repository file:** `src/analyze_data.py`

Produces descriptive statistics, endpoint comparisons, and the World renewable-share trend model with diagnostics.

In [ ]:
print((SRC_DIR / "analyze_data.py").read_text(encoding="utf-8"))

## 10. Figure Generation

**Original repository file:** `src/make_figures.py`

Generates the project's original figures for global energy supply, energy mix, country comparisons, and model diagnostics.

In [ ]:
print((SRC_DIR / "make_figures.py").read_text(encoding="utf-8"))

## 11. Pipeline Execution

**Original repository file:** `src/run_pipeline.py`

Runs the complete workflow from data acquisition and curation through database loading, statistical analysis, and figure generation.

In [ ]:
print((SRC_DIR / "run_pipeline.py").read_text(encoding="utf-8"))

## 12. Notebook Generation

**Original repository file:** `src/build_notebook.py`

Builds the original project analysis notebook from the generated results and figures.

In [ ]:
print((SRC_DIR / "build_notebook.py").read_text(encoding="utf-8"))

## 13. Automated Validation Tests

**Original repository file:** `tests/test_outputs.py`

Validates source-to-fact consistency, analytical and model outputs, descriptive statistics, and generated figures.

In [ ]:
print((TESTS_DIR / "test_outputs.py").read_text(encoding="utf-8"))

# Part II — Core Data and Results

## 14. Pipeline Execution and Validation

PostgreSQL is the primary database. SQLite is an alternative for local analysis reproduction. The pipeline runs as a separate command and writes engine-specific evidence and SQL outputs to the checkout in which it runs. Use a separate checkout if you need both sets of outputs at once.

In [ ]:
required = ("database_run.json", "analysis_panel.csv", "world_series.csv", "countries_series.csv", "annual_coverage.csv")
if any(not (RESULTS_DIR / filename).is_file() for filename in required):
    raise RuntimeError("Run the PostgreSQL or SQLite pipeline from the repository root before executing the notebook.")
database_run = json.loads((RESULTS_DIR / "database_run.json").read_text(encoding="utf-8"))
display(database_run)

## 15. Processed Relational Tables

The curated data are organized into four relational tables:

- `source_release` — identifies the fixed OWID release.
- `geography` — stores geographic identifiers and classifications.
- `indicator` — defines the seven energy measures.
- `energy_observation` — stores geography-year-indicator observations.

Together, they separate source metadata, geographic information, indicator definitions, and geography-year-indicator observations.

In [ ]:
source_release = pd.read_csv(
    PROCESSED_DIR / "source_release.csv"
)

geography = pd.read_csv(
    PROCESSED_DIR / "geography.csv"
)

indicator = pd.read_csv(
    PROCESSED_DIR / "indicator.csv"
)

energy_observation = pd.read_csv(
    PROCESSED_DIR / "energy_observation.csv"
)

print("TABLE DIMENSIONS")
print("-" * 50)
print(f"source_release:     {source_release.shape}")
print(f"geography:          {geography.shape}")
print(f"indicator:          {indicator.shape}")
print(f"energy_observation: {energy_observation.shape}")

In [ ]:
print("SOURCE RELEASE")
display(source_release)

print("\nGEOGRAPHY")
display(geography.head(10))

print("\nINDICATORS")
display(indicator)

print("\nENERGY OBSERVATIONS")
display(energy_observation.head(10))

## 16. Data Quality and Validation Results

The curated dataset distinguishes missing observations from reported zeros and validates relational keys before analysis.

In [ ]:
data_quality = pd.read_csv(
    RESULTS_DIR / "data_quality_summary.csv"
)

display(data_quality)

In [ ]:
key_columns = [
    "release_id",
    "geo_id",
    "year",
    "indicator_id"
]

duplicate_keys = energy_observation.duplicated(
    subset=key_columns
).sum()

print("Independent validation")
print("-" * 50)
print(
    "Total observation slots:",
    f"{len(energy_observation):,}"
)
print(
    "Reported values:",
    f"{energy_observation['value'].notna().sum():,}"
)
print(
    "Missing values:",
    f"{energy_observation['value'].isna().sum():,}"
)
print(
    "Reported zeros:",
    f"{(energy_observation['value'] == 0).sum():,}"
)
print(
    "Duplicate composite keys:",
    f"{duplicate_keys:,}"
)

## 17. SQL Query Results

The database queries produce the World energy series, selected-country energy series, and annual coverage data used in the analysis.

In [ ]:
world_series = pd.read_csv(
    RESULTS_DIR / "world_series.csv"
)

countries_series = pd.read_csv(
    RESULTS_DIR / "countries_series.csv"
)

annual_coverage = pd.read_csv(
    RESULTS_DIR / "annual_coverage.csv"
)

print("SQL EXPORT DIMENSIONS")
print("-" * 50)

print("World series:", world_series.shape)
print("Country series:", countries_series.shape)
print("Annual coverage:", annual_coverage.shape)

In [ ]:
print("WORLD SERIES")
display(world_series.head(10))

print("\nSELECTED-COUNTRY SERIES")
display(countries_series.head(10))

print("\nANNUAL COVERAGE")
display(annual_coverage.head(10))

## 18. Common 2000–2025 Analytical Panel

The primary analysis uses a balanced 2000–2025 panel covering the World, China, India, Germany, and the United States. All seven selected indicators are available across these five locations for the full 26-year period.

In [ ]:
analysis_panel = pd.read_csv(
    RESULTS_DIR / "analysis_panel.csv"
)

print("Analytical panel shape:", analysis_panel.shape)
print(
    "Year range:",
    analysis_panel["year"].min(),
    "to",
    analysis_panel["year"].max()
)

print(
    "Locations:",
    sorted(analysis_panel["name"].unique())
)

print(
    "Indicators:",
    analysis_panel["indicator_id"].nunique()
)

display(analysis_panel.head(10))

## 19. Descriptive Statistics

Descriptive statistics summarize the distribution of each energy indicator across the five study locations over the common 2000–2025 period.

In [ ]:
descriptive_statistics = pd.read_csv(
    RESULTS_DIR / "descriptive_statistics.csv"
)

display(descriptive_statistics)

## 20. Changes Between 2000 and 2025

Endpoint comparisons summarize how total energy supply and energy-source shares changed across the study locations between 2000 and 2025.

In [ ]:
endpoints = pd.read_csv(
    RESULTS_DIR / "endpoints.csv"
)

display(endpoints)

## 21. World Total Energy Supply

The world total energy supply is examined over the common 2000–2025 analytical period.

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            FIGURES_DIR / "01_world_total.png"
        ),
        width=850
    )
)

## 22. World Energy Mix

The World Energy Mix compares the shares of coal, oil, natural gas, renewables, and nuclear energy from 2000 through 2025.

In [ ]:
display(
    Image(
        filename=str(
            FIGURES_DIR / "02_world_mix.png"
        ),
        width=850
    )
)

## 23. Selected-Country Comparisons

Total energy supply and fossil-fuel share are compared across China, India, Germany, and the United States over the 2000–2025 study period.

In [ ]:
display(
    Image(
        filename=str(
            FIGURES_DIR / "03_countries.png"
        ),
        width=900
    )
)

## 24. World Renewable-Share Trend Model

An ordinary least-squares trend summarizes the average change in World renewable-energy share from 2000 through 2025. The model is descriptive rather than predictive, and residual diagnostics are used to assess temporal dependence and departures from a simple linear trend.

In [ ]:
model_statistics = pd.read_csv(
    RESULTS_DIR / "model_statistics.csv"
)

model_diagnostics = pd.read_csv(
    RESULTS_DIR / "model_diagnostics.csv"
)

display(model_statistics)

print("\nModel diagnostic observations:")
display(model_diagnostics.head(10))

## 25. Renewable-Share Model Diagnostics

Observed and fitted renewable-energy shares are compared with residual patterns to evaluate the descriptive linear trend.

In [ ]:
display(
    Image(
        filename=str(
            FIGURES_DIR / "04_model_diagnostics.png"
        ),
        width=850
    )
)

## 26. Data-Management Workflow and Database Design

**Original repository file:** `docs/data_flow.mmd`

The workflow diagram summarizes the project from the pinned OWID source through profiling, curation, database storage, SQL retrieval, statistical analysis, and final outputs.

In [ ]:
data_flow_mermaid = (
    DOCS_DIR / "data_flow.mmd"
).read_text(encoding="utf-8")

print(data_flow_mermaid)

In [ ]:
display(Image(filename=str(DOCS_DIR / "data_flow.png"), width=800))

### Relational Database Model

**Original repository file:** `docs/data_model.mmd`

The database model shows the relationships among 
- source_release,
- geography,
- indicator, and
- energy_observation.

In [ ]:
data_model_mermaid = (
    DOCS_DIR / "data_model.mmd"
).read_text(encoding="utf-8")

print(data_model_mermaid)

In [ ]:
display(Image(filename=str(DOCS_DIR / "data_model.png"), width=800))

### Data and Database Documentation

**Original repository file:** `docs/DATA_AND_DATABASE.md`

The following output preserves the team's technical documentation of the source data, cleaning workflow, database design, and analytical data-management process.

In [ ]:
print(
    (DOCS_DIR / "DATA_AND_DATABASE.md")
    .read_text(encoding="utf-8")
)

### Reproducibility Documentation

**Original repository file:** `docs/APPENDIX_REPRODUCIBILITY.md`

Documents the project's data sources, curation process, analysis workflow, reproduction steps, and key limitations.

In [ ]:
print(
    (DOCS_DIR / "APPENDIX_REPRODUCIBILITY.md")
    .read_text(encoding="utf-8")
)

### Source Snapshot Metadata

**Original repository file:** `docs/source_snapshot.json`

Records the source URL, release version, file size, checksum, and acquisition and verification timestamps for the fixed OWID dataset.

In [ ]:
source_snapshot = json.loads(
    (DOCS_DIR / "source_snapshot.json")
    .read_text(encoding="utf-8")
)

display(source_snapshot)

# Part III — Geography and Coverage

## 29. Source Grid and ISO Codes

The geographic panel has one row per year and ISO-coded OWID location, including territories and historical labels. A reported value includes a recorded zero. Missing means OWID supplies a location-year row with no selected total-supply value. Outside source grid means OWID supplies no row for that pair. The third state is outside the assessment of reporting and does not infer the dates of political existence. The modern basemap cannot reconstruct earlier boundaries.

In [ ]:
geography_panel = geographic_panel(geography, energy_observation, MAP_START, int(energy_observation.year.max()))
coverage_by_year = geographic_coverage(geography_panel)
no_shape = geography_panel.loc[~geography_panel.iso_code.isin(map_codes()), ["iso_code", "name"]]
no_shape = no_shape.drop_duplicates().sort_values("iso_code")
print(f"ISO-coded locations in tabular results: {geography_panel.iso_code.nunique()}")
print(f"Matched to the Plotly 50m basemap: {geography_panel.iso_code.nunique() - len(no_shape)}")
display(no_shape)
display(geography_panel.loc[geography_panel.year.eq(MAP_END)].head())

## 30. Renewable Share Maps

The snapshot and animation use a fixed 0–100% scale and show reported renewable share. Hover over a shaded location to see its energy profile. A location can have a reported total with a missing renewable share. View the maps in a browser with access to the Plotly and basemap CDNs. The coverage table and static chart remain available without the interactive map.

In [ ]:
latest, renewable_animation, availability, coverage_trend = geographic_figures(geography_panel, coverage_by_year)
latest.show(renderer="notebook_connected")
renewable_animation.show(renderer="notebook_connected")

## 31. Supply Availability and Coverage

The map uses current boundaries and shows three statuses for total supply. The coverage chart and table include every ISO-coded location in the data, including those without a matching polygon. The broad maps end in 2024 because the source reports more total-supply values that year. The core comparison runs through 2025 because all seven indicators are reported for each of the five study locations.

In [ ]:
availability.show(renderer="notebook_connected")
coverage_trend.show(renderer="notebook_connected")
display(coverage_by_year.loc[coverage_by_year.year.isin([1965, 1970, 1980, 1990, 2000, 2010, 2020, 2024, 2025])])

# Part IV — Historical World Energy Supply

## 32. Component Coverage and Reconciliation

This extension reads nine disjoint source quantities in TWh from the same pinned file. Fossil-fuel, renewable, and low-carbon aggregates are excluded from the stack because they overlap its components. Sums require every component, and recorded zeros are valid. The renewable sum is checked against the published renewable aggregate using a 0.001 TWh tolerance. The nine-source sum is compared separately with published total supply, which is displayed at its original value.

In [ ]:
raw_energy = pd.read_csv(RAW_DIR / "owid_energy.csv")
history = historical_values(raw_energy)
observed = history.dropna(subset=["component_sum_twh", TOTAL])
renewable_gaps = history.renewable_difference_twh.dropna().abs()
peak_year = observed.total_difference_twh.abs().idxmax()
peak_difference = observed.loc[peak_year, "total_difference_twh"]
print(f"Stored World rows: {len(history)}. Years with all nine quantities and published total: {len(observed)}.")
print(f"Largest absolute renewable gap: {renewable_gaps.max():.9f} TWh.")
print(f"Largest total difference: {peak_difference:+.6f} TWh in {peak_year}.")
assert renewable_gaps.max() <= RENEWABLE_TOLERANCE_TWH
display(observed[["renewable_sum_twh", "renewable_difference_twh", "component_sum_twh", TOTAL,
                  "total_difference_twh"]].tail(10))

## 33. Historical Supply by Source

The stacked bars show only measured years. Observations are decadal from 1800 through 1960 and annual from 1965 onward. The dots show published total supply. The component-to-total difference is retained and its exact cause is unresolved. Each animation step adds an observed year.

In [ ]:
historical_chart, historical_animation = historical_figures(history)
historical_chart.show(renderer="notebook_connected")
historical_animation.show(renderer="notebook_connected")
make_extension_figures(coverage_by_year, history)
for filename in ("coverage.png", "historical_supply.png"):
    display(Image(filename=str(FIGURES_DIR / "extended" / filename), width=800))

## 34. Scope and Interpretation

The core panel summarizes shifts in amounts and shares for five fixed study locations. Maps describe reporting coverage and geographic variation, and historical bars describe source quantities in observed World years. Changes in geographic coverage, current boundaries, earlier sparse measurements, the OWID accounting convention, and the serially dependent regression residuals limit interpretation. The [data guide](../docs/DATA_AND_DATABASE.md) records attribution and upstream reuse terms.